### 01. 编译无参的prim_func(报错)

In [ ]:
import tilelang
import tilelang.language as T

@T.prim_func
def main(
    # A: T.Tensor((1,1), "float32"), # type: ignore
):
    pass

try:
    jit_kernel = tilelang.compile(main)
    kernel_source = jit_kernel.get_kernel_source()
    print(kernel_source)
except Exception as e:
    pass

### 02. 编译有参的prim_func(Add为例)

In [ ]:

import tilelang
import tilelang.language as T

@T.prim_func
def main(
    A: T.Tensor((1,1), "float32"), # type: ignore
    B: T.Tensor((1,1), "float32"), # type: ignore
    C: T.Tensor((1,1), "float32"), # type: ignore
):
    with T.Kernel(1, is_npu=True) as (cid, vid):
        a_ub = T.alloc_ub((1,1), "float32")
        b_ub = T.alloc_ub((1,1), "float32")
        c_ub = T.alloc_ub((1,1), "float32")

        T.copy(A[0, 0], a_ub)
        T.copy(B[0, 0], b_ub)

        T.tile.add(c_ub, a_ub, b_ub)

        T.copy(c_ub, C[0, 0])

try:
    jit_kernel = tilelang.compile(main, pass_configs={
        tilelang.PassConfigKey.TL_ASCEND_AUTO_SYNC: True,
    })
    kernel_source = jit_kernel.get_kernel_source()
    print(kernel_source)
except Exception as e:
    pass